# **Spam Ham Classifier**
## with lemmatization

In [1]:
import numpy
import pandas as pd
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv("hf://datasets/thehamkercat/telegram-spam-ham/dataset.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20348 entries, 0 to 20347
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text_type  20348 non-null  object
 1   text       20348 non-null  object
dtypes: object(2)
memory usage: 318.1+ KB


In [4]:
df.head(3)

,text_type,text
0,spam,naturally irresistible your corporate identity...
1,spam,the stock trading gunslinger fanny is merrill ...
2,spam,unbelievable new homes made easy im wanting to...


In [5]:
df.shape

(20348, 2)

In [6]:
df.isna().sum()

text_type    0
text         0
dtype: int64

In [7]:
df.duplicated().sum()

14

In [8]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True,inplace=True)

In [9]:
df.text_type.value_counts()

text_type
ham     14323
spam     6011
Name: count, dtype: int64

In [10]:
df=df.rename(columns={'text_type':'target'})

In [11]:
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
stop_words=set(stopwords.words('english'))

from nltk.tokenize import word_tokenize

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:

import re
def clean(word):
    word=word.lower()
    word = re.sub(r"^[^A-Za-z0-9₹$€£¥]+|[^A-Za-z0-9\s₹$€£¥]+|[\s]+$", "", word).strip()  #only keeping digits,alphabet,currency symbol
    tokens=word_tokenize(word)
    tokens_without_stopwords=[x for x in tokens if x not in stop_words]
    cleaned_text = " ".join(tokens_without_stopwords)

    return cleaned_text

df.text=df.text.apply(clean)

In [13]:
df.text

0        naturally irresistible corporate identity lt r...
1        stock trading gunslinger fanny merrill muzo co...
2        unbelievable new homes made easy im wanting sh...
3        4 color printing special request additional in...
4        money get software cds software compatibility ...
                               ...                        
20329    spam alert user username dillybubbles 91914695...
20330                                             hum ky h
20331                                                  ban
20332                                            kaisi hii
20333                                              shock q
Name: text, Length: 20334, dtype: object

In [14]:
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk import pos_tag

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('V'):
        return 'v'
    elif tag.startswith('N'):
        return 'n'
    elif tag.startswith('R'):
        return 'r'
    else:
        return 'n'

# Function to process a single sentence
def lemmatize_text(sentence):
    tokens = word_tokenize(sentence)
    pos_tags = pos_tag(tokens)
    
    lemmatized = [
        lemmatizer.lemmatize(word, get_wordnet_pos(pos))
        for word, pos in pos_tags
    ]
    
    return " ".join(lemmatized)

# Apply to dataframe column
df["lemmatized_text"] = df["text"].apply(lemmatize_text)

df["lemmatized_text"]

0        naturally irresistible corporate identity lt r...
1        stock trading gunslinger fanny merrill muzo co...
2        unbelievable new home make easy im want show h...
3        4 color printing special request additional in...
4        money get software cd software compatibility g...
                               ...                        
20329    spam alert user username dillybubbles 91914695...
20330                                             hum ky h
20331                                                  ban
20332                                            kaisi hii
20333                                              shock q
Name: lemmatized_text, Length: 20334, dtype: object

In [15]:
df.head()

,target,text,lemmatized_text
0,spam,naturally irresistible corporate identity lt r...,naturally irresistible corporate identity lt r...
1,spam,stock trading gunslinger fanny merrill muzo co...,stock trading gunslinger fanny merrill muzo co...
2,spam,unbelievable new homes made easy im wanting sh...,unbelievable new home make easy im want show h...
3,spam,4 color printing special request additional in...,4 color printing special request additional in...
4,spam,money get software cds software compatibility ...,money get software cd software compatibility g...


In [16]:
x=df['lemmatized_text']
y=df['target']

In [17]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.25,random_state=42)

In [18]:
from sklearn.preprocessing import LabelEncoder
LE=LabelEncoder()
y_train=LE.fit_transform(y_train)
y_test=LE.transform(y_test)

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizor = TfidfVectorizer(max_features=5000)
x_train = vectorizor.fit_transform(x_train)
x_test = vectorizor.transform(x_test)

In [20]:
#handling imbalancing in data
#resampling on test data doesn't happen

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

x_train_resampled, y_train_resampled = smote.fit_resample(x_train, y_train)

In [21]:
x_train.shape

(15250, 5000)

In [22]:
pd.DataFrame(y_train_resampled).value_counts()

0
0    10684
1    10684
Name: count, dtype: int64

In [23]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
model.fit(x_train_resampled,y_train_resampled)

MultinomialNB()

In [24]:
y_pred_test=model.predict(x_test)

In [25]:
# model accuracy
from sklearn.metrics import accuracy_score,classification_report
accuracy_score(y_test,y_pred_test)

0.8878835562549174

In [26]:
print(classification_report(y_test,y_pred_test))

              precision    recall  f1-score   support

           0       0.96      0.88      0.92      3639
           1       0.75      0.90      0.82      1445

    accuracy                           0.89      5084
   macro avg       0.86      0.89      0.87      5084
weighted avg       0.90      0.89      0.89      5084



0=ham

1=spam

In [30]:
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('V'):
        return 'v'
    elif tag.startswith('N'):
        return 'n'
    elif tag.startswith('R'):
        return 'r'
    else:
        return 'n'
    
def predict(word):
    word=word.lower()
    word = re.sub(r"^[^A-Za-z0-9₹$€£¥]+|[^A-Za-z0-9\s₹$€£¥]+|[\s]+$", "", word).strip()  #only keeping digits,alphabet,currency symbol
    tokens=word_tokenize(word)
    tokens_without_stopwords=[x for x in tokens if x not in stop_words]
    # cleaned_text = " ".join(tokens_without_stopwords)

    # tokens = word_tokenize(cleaned_text)
    pos_tags = pos_tag(tokens_without_stopwords)
    
    lemma = [
        lemmatizer.lemmatize(word, get_wordnet_pos(pos))
        for word, pos in pos_tags
    ]
    
    vector=vectorizor.transform(lemma)
    prediction=model.predict(vector)
    
    if prediction[0]==0:
        return "ham"
    else:
        return "spam"
    
    

In [31]:
predict("winning $2000")

'spam'

In [29]:
LE.classes_

array(['ham', 'spam'], dtype=object)

# improvements:
run all base model

find best base model

include cv maybe

tune the best model

try other vectorizer

try different max_feature